# Teaching a small model to write SQL with GRPO

**Scenario.** Store managers and analysts at Taco Alley want to ask questions in English and get
answers out of the operational database. A model that emits *plausible-looking* SQL is worse than
useless here - a query that runs but silently answers the wrong question is a bad number in a
business review.

**Why GRPO fits.** Group Relative Policy Optimization (GRPO) needs one thing: a function that scores a
completion. It does not need a reward model, and it does not need a human-written chain of
reasoning for every example. Text-to-SQL hands us a reward for free - *run the query and compare
the result set to the gold query's result set*. That is a verifiable reward, which is exactly the
regime where GRPO earns its keep.

The loop, concretely:

1. Sample `G` candidate queries for the same question (the *group*).
2. Execute each one against a read-only copy of the database.
3. Score each: does it parse, does it run, does it return the right rows?
4. Center each completion's reward against its group's mean. With the default group scaling, also divide by the group's reward standard deviation plus a small epsilon. Better-than-average candidates get reinforced, worse-than-average get suppressed. No value network, no critic.

Equal-reward groups provide no reward-based policy-gradient signal; check whether this reflects consistent success, consistent failure, or insufficient reward discrimination.

That third step is where all the engineering actually lives, and it is where most of this notebook
goes.

---

## What you need

| Resource | Guidance |
| --- | --- |
| GPU | A BF16-capable GPU for a 1.5B policy with low-rank adaptation (LoRA); validate memory use on your selected hardware. |
| Time | Runtime is hardware- and configuration-dependent. This example uses 367 prompts, two epochs, and 12 generations per group without enabling vLLM; benchmark a short run to estimate total time. |
| Files | `data/taco_alley.db`, `data/train.jsonl`, `data/eval_*.jsonl` (all in the datasets directory) |

These settings are a starting point, not a guaranteed fit in 16 GB VRAM. Memory and runtime depend on the model, precision, optimizer, batch size, sequence length, and evaluation workload. Reduce the batch size or use a smaller model when needed. Validate a short run on your selected GPU before committing to a full training run.

Generation, not backprop, is the bottleneck. Watch `step_time` in the logs.

<a href="https://colab.research.google.com/github/ned1313/Fine-tuning-and-Optimizing-Small-Language-Models/blob/main/notebooks/grpo_lora_run.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Google Colab prep

If you are running this notebook in Google Colab, select a GPU runtime and run the code block below to install the necessary packages. If you are running locally with the project dependencies already installed, skip this cell.

The current code uses BF16 on the GPU. Select a BF16-capable runtime, such as a supported Ampere-or-newer Nvidia GPU, or explicitly adapt and test the precision settings before using a T4.

The install cell installs packages only. Clone or upload the repository into the Colab runtime, including its datasets, and set the kernel's working directory to the repository's `notebooks` directory before continuing. This also keeps relative data, credential, and artifact paths consistent with local runs. Preserve any generated artifacts you need before the runtime is discarded.

For local runs, also set the kernel's working directory to the repository's `notebooks` directory. Opening the notebook in an editor does not by itself guarantee that working directory.

Optional: vLLM can accelerate rollouts on supported runtimes. Install it separately with `%pip install "trl[vllm]"` if you plan to enable it.

In [ ]:
%pip install "trl>=1.0" "peft>=0.14" "transformers>=4.55" "datasets>=3.0" "accelerate>=1.0" "bitsandbytes>=0.45" sqlglot matplotlib

In [ ]:
import json, os, re, sqlite3, threading, time
from functools import lru_cache
from pathlib import Path

import torch
from datasets import Dataset
from transformers import AutoTokenizer

REPO_ROOT = Path.cwd().resolve().parent
DATASETS_DIR = REPO_ROOT / "datasets"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
DB_PATH = DATASETS_DIR / "taco_alley.db"
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B"   # strong SQL prior for its size
OUTPUT_DIR = ARTIFACTS_DIR / "grpo-taco-alley"

# The database is frozen in time. Anything relative ("last month") is resolved
# against this date, and the model is told what it is.
TODAY = "2026-08-24"

assert DB_PATH.exists(), "Place taco_alley.db in datasets/"

## The database, and a read-only handle on it

The reward function executes *model-generated* SQL. That is untrusted input by definition, so the
sandbox matters even in a demo:

- connections are opened with `mode=ro` **and** `PRAGMA query_only`, so a `DROP TABLE` that slips
  past the text filter still cannot land;
- a progress handler aborts any query that burns more than a fixed number of VM steps, which kills
  runaway cartesian joins - a model that has not learned join conditions yet will absolutely write
  one;
- one connection per thread, since the trainer may score completions off the main thread.

In [ ]:
_local = threading.local()
# Calibrate the budget ABOVE your slowest gold query, or you will silently abort
# valid SQL and hand the model a zero reward it did not earn. The heaviest gold
# query in this dataset costs ~13M VM steps / 0.9s.
QUERY_STEP_BUDGET = 50_000_000
QUERY_TIMEOUT_S   = 2.0


def get_conn() -> sqlite3.Connection:
    conn = getattr(_local, "conn", None)
    if conn is None:
        conn = sqlite3.connect(f"{DB_PATH.as_uri()}?mode=ro", uri=True, check_same_thread=False)
        conn.execute("PRAGMA query_only = ON")
        _local.conn = conn
    return conn


def execute_sql(sql: str, max_rows: int = 200):
    """Run one read-only statement under a step + wall-clock budget."""
    conn = get_conn()
    deadline = time.time() + QUERY_TIMEOUT_S
    state = {"steps": 0}

    def guard():
        state["steps"] += 1
        if state["steps"] > QUERY_STEP_BUDGET or time.time() > deadline:
            return 1          # non-zero aborts the query
        return 0

    conn.set_progress_handler(guard, 1000)
    try:
        cur = conn.execute(sql)
        return cur.fetchmany(max_rows)
    finally:
        conn.set_progress_handler(None, 0)


# smoke test - the question from the original brief
print(execute_sql("""
    SELECT COUNT(*) AS complaint_count
      FROM customer_tickets
     WHERE store_name = 'Springfield'
       AND category = 'Complaint'
       AND created_at >= '2026-07-01'
       AND created_at < '2026-08-01'
"""))

## The prompt

The whole schema goes in the system prompt and with enum hints. This gives the model a good idea of the database structure and standard values it can query for, like stores in Pennsylvania should use `PA` and not the full state name.

In [ ]:
ENUM_HINTS = {
    ("stores", "state"):              ["PA", "NJ", "DE"],
    ("stores", "region"):             ["Northeast", "Mid-Atlantic"],
    ("employees", "role"):            ["Crew", "Shift Lead", "Assistant Manager", "General Manager"],
    ("menu_items", "category"):       ["Taco", "Burrito", "Bowl", "Side", "Drink", "Dessert"],
    ("loyalty_members", "tier"):      ["Bronze", "Silver", "Gold"],
    ("orders", "channel"):            ["In-Store", "Drive-Thru", "Mobile App", "Delivery"],
    ("orders", "payment_method"):     ["Card", "Cash", "Mobile Wallet", "Gift Card"],
    ("orders", "status"):             ["Completed", "Refunded", "Cancelled"],
    ("customer_tickets", "channel"):  ["Phone", "Email", "App", "In-Person", "Social"],
    ("customer_tickets", "category"): ["Complaint", "Compliment", "Question", "Refund Request"],
    ("customer_tickets", "priority"): ["Low", "Medium", "High"],
    ("customer_tickets", "status"):   ["Open", "In Progress", "Resolved", "Closed"],
}

def render_schema() -> str:
    conn = get_conn()
    tables = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name")]
    out = []
    for t in tables:
        cols = []
        for _, name, ctype, _, _, pk in conn.execute(f"PRAGMA table_info({t})"):
            bit = f"{name} {ctype}"
            if pk:
                bit += " PK"
            if (t, name) in ENUM_HINTS:
                bit += " in (" + ", ".join(repr(v) for v in ENUM_HINTS[(t, name)]) + ")"
            cols.append(bit)
        out.append(f"{t}(\n  " + ",\n  ".join(cols) + "\n)")
    fks = []
    for t in tables:
        for row in conn.execute(f"PRAGMA foreign_key_list({t})"):
            fks.append(f"{t}.{row[3]} -> {row[2]}.{row[4]}")
    return "\n".join(out) + "\n\nForeign keys:\n" + "\n".join(f"  {f}" for f in fks)

SCHEMA = render_schema()

SYSTEM_PROMPT = f"""You are a SQL analyst for Taco Alley, a taco restaurant chain.
Convert the user's question into a single SQLite SELECT statement.

Schema:
{SCHEMA}

Rules:
- Today's date is {TODAY}. Resolve relative dates against it.
- Timestamps are TEXT in 'YYYY-MM-DD HH:MM:SS' form; dates are 'YYYY-MM-DD'. Half-open ranges
  (>= start AND < end) are the safe way to filter a month.
- Revenue and sales questions mean orders with status = 'Completed' unless stated otherwise.
- Return exactly one SELECT statement. No INSERT, UPDATE, DELETE, or DDL.
- Reply with only the query inside a ```sql fenced block. No explanation."""

tok = AutoTokenizer.from_pretrained(MODEL_ID)
print(SCHEMA[:600], "...\n")
print("system prompt tokens:", len(tok(SYSTEM_PROMPT)["input_ids"]))

## Dataset

Three splits ship with the demo:

- `train.jsonl` - 367 question/gold-SQL pairs
- `eval_params.jsonl` - same question shapes, parameter values the model never trained on
- `eval_templates.jsonl` - question shapes held out entirely

The second eval split is the honest one. Improving on `eval_params` mostly shows the model learned
the templates; improving on `eval_templates` is the claim that it learned to read the schema. Report
both.

Note what the dataset does *not* contain: reasoning traces, or even a single example of the model's
own output. GRPO generates its own training signal. `gold_sql` is only ever used inside the reward
function, never shown to the model.

In [ ]:
def load_split(name: str) -> Dataset:
    with (DATASETS_DIR / f"{name}.jsonl").open(encoding="utf-8") as data_file:
        rows = [json.loads(line) for line in data_file]
    for row in rows:
        row["prompt"] = [{"role": "system", "content": SYSTEM_PROMPT},
                         {"role": "user", "content": row["question"]}]
    return Dataset.from_list(rows)


train_ds = load_split("train")
eval_params = load_split("eval_params")
eval_template = load_split("eval_templates")

print({key: len(value) for key, value in
       dict(train=train_ds, eval_params=eval_params, eval_templates=eval_template).items()})
print("\nexample question:", train_ds[0]["question"])
print("gold:", train_ds[0]["gold_sql"][:160])

## Scoring a completion

Four things to get right before writing any reward:

**Extraction.** The model wraps its answer in a fenced block. Pull it out; if there is no fence,
fall back to the first `SELECT`/`WITH` in the text. Be generous here - punishing formatting twice
(once in the format reward, once by failing to extract) makes the reward needlessly sparse.

**Safety.** Reject anything that is not a single read-only statement, *before* execution.

**Result comparison.** Exact string match on SQL is the wrong metric - `COUNT(*)` and
`COUNT(1)` are the same query. Compare result sets instead. Two details:

- if the gold query has an `ORDER BY`, row order is part of the answer, so compare ordered;
  otherwise compare as a multiset;
- round floats before comparing, or `8163.369999999999` fails against `8163.37`.

**Partial credit.** A pure right/wrong reward can be sparse. GRPO centers each completion's reward against its group's mean and, with default group scaling, divides by the group's reward standard deviation plus a small epsilon. Equal-reward groups provide no reward-based policy-gradient signal. If `frac_reward_zero_std` stays near 1.0, inspect actual rewards to distinguish consistent success, consistent failure, and insufficient reward discrimination. The overlap term below provides partial credit for nearly correct results.

In [ ]:
FENCE_RE  = re.compile(r"```(?:sql)?\s*(.+?)```", re.S | re.I)
SELECT_RE = re.compile(r"\b(SELECT|WITH)\b.*", re.S | re.I)
FORBIDDEN = re.compile(
    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|REPLACE|TRUNCATE|ATTACH|DETACH|PRAGMA|VACUUM)\b", re.I)

def extract_sql(text: str) -> str | None:
    m = FENCE_RE.search(text)
    candidate = m.group(1) if m else None
    if candidate is None:
        m = SELECT_RE.search(text)
        candidate = m.group(0) if m else None
    if candidate is None:
        return None
    return candidate.strip().rstrip(";").strip()

def is_single_read_only(sql: str) -> bool:
    if not sql or FORBIDDEN.search(sql):
        return False
    if ";" in sql.strip().rstrip(";"):      # more than one statement
        return False
    return bool(re.match(r"\s*(SELECT|WITH)\b", sql, re.I))

def canon(value):
    if isinstance(value, float):
        return round(value, 2)
    if isinstance(value, int):
        return float(value)                  # 17 == 17.0 across COUNT vs SUM
    if isinstance(value, str):
        return value.strip()
    return value

def normalize(rows, ordered: bool):
    norm = [tuple(canon(v) for v in row) for row in rows]
    return tuple(norm) if ordered else tuple(sorted(norm, key=repr))

@lru_cache(maxsize=8192)
def result_signature(sql: str, ordered: bool):
    """(status, normalized_rows). status is 'ok' | 'unsafe' | 'error'.

    Cached: the four reward functions ask for the same gold result once per
    completion, and every member of a group shares one gold. Without the cache
    a group of 8 executes the gold query 16 times instead of once."""
    if not is_single_read_only(sql):
        return "unsafe", None
    try:
        return "ok", normalize(execute_sql(sql), ordered)
    except Exception:
        # a query killed by the progress handler arrives as OperationalError
        return "error", None

def row_overlap(pred, gold) -> float:
    """Jaccard over rows - dense-ish partial credit for a nearly-right query."""
    if not pred or not gold:
        return 0.0
    a, b = {repr(r) for r in pred}, {repr(r) for r in gold}
    return len(a & b) / len(a | b)

### The reward functions

Four separate functions rather than one, because TRL logs `rewards/<name>/mean` per function. That per-component breakdown helps diagnose a run. If `executes` climbs while `result_match` stays flat, the model may be producing executable SQL that answers the wrong question. Inspect generated SQL, component rewards, data, prompt, and optimization settings before choosing a change.

Weights (`[0.15, 0.25, 1.0, 0.25]`, set in the config below) keep correctness dominant. The `format` function is the only one that can go negative: it penalizes extracted statements that fail the read-only check.

In [ ]:
def _pairs(completions, gold_sql):
    """TRL passes conversational completions as [{'role':..., 'content':...}]."""
    texts = [c[0]["content"] if isinstance(c, list) else c for c in completions]
    return list(zip(texts, gold_sql))

def reward_format(completions, gold_sql, **kw):
    out = []
    for text, _ in _pairs(completions, gold_sql):
        sql = extract_sql(text)
        if sql is None:
            out.append(0.0)
        elif not is_single_read_only(sql):
            out.append(-1.0)                      # tried to mutate, or emitted junk
        elif FENCE_RE.search(text):
            out.append(1.0)                       # fenced, single, read-only
        else:
            out.append(0.5)                       # right query, sloppy wrapper
    return out

def reward_executes(completions, gold_sql, **kw):
    out = []
    for text, _ in _pairs(completions, gold_sql):
        sql = extract_sql(text) or ""
        status, _rows = result_signature(sql, ordered=False)
        out.append(1.0 if status == "ok" else 0.0)
    return out

def reward_result_match(completions, gold_sql, log_metric=None, **kw):
    out = []
    for text, gold in _pairs(completions, gold_sql):
        ordered = bool(re.search(r"\bORDER\s+BY\b", gold, re.I))
        g_status, g_rows = result_signature(gold, ordered)
        p_status, p_rows = result_signature(extract_sql(text) or "", ordered)
        out.append(1.0 if (p_status == "ok" and g_status == "ok" and p_rows == g_rows) else 0.0)
    if log_metric:
        log_metric("exec_accuracy", sum(out) / max(len(out), 1))
    return out

def reward_row_overlap(completions, gold_sql, **kw):
    out = []
    for text, gold in _pairs(completions, gold_sql):
        _g, g_rows = result_signature(gold, ordered=False)
        p_status, p_rows = result_signature(extract_sql(text) or "", ordered=False)
        out.append(row_overlap(p_rows, g_rows) if p_status == "ok" else 0.0)
    return out

REWARD_FUNCS   = [reward_format, reward_executes, reward_result_match, reward_row_overlap]
REWARD_WEIGHTS = [0.15,          0.25,            1.0,                 0.25]

### Sanity-check the reward before you spend a GPU-hour on it

Reward bugs are silent. A function that returns 1.0 for everything trains a model to do nothing in
particular, and the loss curve will look fine while it happens. Score a few completions by hand
first - including the cases you expect to be adversarial.

In [ ]:
gold = ("SELECT COUNT(*) AS complaint_count FROM customer_tickets "
        "WHERE store_name = 'Springfield' AND category = 'Complaint' "
        "AND created_at >= '2026-07-01' AND created_at < '2026-08-01'")

probes = {
    "gold, reformatted":  "```sql\nSELECT COUNT(1) FROM customer_tickets WHERE store_name='Springfield'"
                          " AND category='Complaint' AND created_at BETWEEN '2026-07-01' AND '2026-07-31 23:59:59'\n```",
    "right idea, wrong literal": "```sql\nSELECT COUNT(*) FROM customer_tickets WHERE store_name='Springfield'"
                          " AND category='complaint' AND created_at >= '2026-07-01' AND created_at < '2026-08-01'\n```",
    "hallucinated column": "```sql\nSELECT COUNT(*) FROM customer_tickets WHERE store = 'Springfield'\n```",
    "no fence":           "SELECT COUNT(*) FROM customer_tickets WHERE store_name='Springfield'"
                          " AND category='Complaint' AND created_at >= '2026-07-01' AND created_at < '2026-08-01'",
    "prose only":         "You could count the rows in the customer_tickets table.",
    "destructive":        "```sql\nDROP TABLE customer_tickets\n```",
}

names = list(probes)
comps = [[{"role": "assistant", "content": probes[n]}] for n in names]
golds = [gold] * len(names)
scores = {f.__name__: f(completions=comps, gold_sql=golds) for f in REWARD_FUNCS}

print(f"{'probe':<28}" + "".join(f"{k.replace('reward_',''):>14}" for k in scores) + f"{'total':>9}")
for i, n in enumerate(names):
    vals = [scores[k][i] for k in scores]
    total = sum(v * w for v, w in zip(vals, REWARD_WEIGHTS))
    print(f"{n:<28}" + "".join(f"{v:>14.2f}" for v in vals) + f"{total:>9.2f}")

Inspect each reward component, not just the total. The wrong-literal query can execute successfully and earn formatting and execution credit while receiving no correctness credit; `COUNT(*)` returns a zero count rather than an empty result set. The destructive probe is rejected before execution and receives a weighted total of -0.15. Gold answers that are zero or empty need special scrutiny because unrelated incorrect queries may return the same result. Audit these cases in the supplied dataset; this repository does not include the dataset-generation script.

## Baseline

Measure before you train, on both eval splits, with greedy decoding. Without this number the
after-training number means nothing.

In [ ]:
from transformers import AutoModelForCausalLM

@torch.no_grad()
def exec_accuracy(model, tokenizer, ds, batch_size=8, max_new_tokens=256):
    hits = []
    tokenizer.padding_side = "left"
    for i in range(0, len(ds), batch_size):
        chunk = ds[i:i + batch_size]
        prompts = [tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
                   for p in chunk["prompt"]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        texts = tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for text, gold in zip(texts, chunk["gold_sql"]):
            ordered = bool(re.search(r"\bORDER\s+BY\b", gold, re.I))
            _g, g_rows = result_signature(gold, ordered)
            p_status, p_rows = result_signature(extract_sql(text) or "", ordered)
            hits.append(1.0 if (p_status == "ok" and p_rows == g_rows) else 0.0)
    return sum(hits) / len(hits)

tok.pad_token = tok.pad_token or tok.eos_token


In [ ]:

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, 
    dtype=torch.bfloat16,
    device_map="cuda",
)
base.eval()

baseline = {"eval_params":    exec_accuracy(base, tok, eval_params),
            "eval_templates": exec_accuracy(base, tok, eval_template)}
print("baseline execution accuracy:", {k: f"{v:.1%}" for k, v in baseline.items()})

In [ ]:
import gc
base = None
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(torch.cuda.memory_allocated())
print(torch.cuda.memory_reserved())


## Train

This single-GPU configuration uses 12 generations, batch size 1, and 12 accumulation steps. The effective generation batch must be compatible with the group size; include all devices and any generation-batch overrides when changing it. Group size and sampling temperature trade generation cost against diversity, so compare short runs rather than treating the current values as universal optima.

- **`num_generations` (the group size, `G`).** More completions provide more opportunities for within-group reward differences, but increase generation cost. Inspect reward diversity when choosing a group size.
- **`max_completion_length`.** This run allows 256 tokens per completion. Check `completions/clipped_ratio` and sampled SQL before reducing the limit; truncation can remove a valid query's ending.
- **`beta` (KL penalty).** This run uses 0.0 and does not use a KL reference term. A nonzero value constrains updates relative to a reference policy and introduces reference-scoring work.
- **`temperature`.** This run uses 1.0. Compare sampling diversity, rewards, and held-out results when changing it.

This notebook uses LoRA on a non-quantized base model. To adapt it to quantized LoRA (QLoRA), configure 4-bit quantization when loading the base model, or use a trainer-supported model-identifier loading path, then verify backend support and training compatibility. Adding `quantization_config` to the trainer does not quantize the model instance already created above. With `beta=0`, this run does not use a KL reference term.

In [ ]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

peft_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules="all-linear",
)

args = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    # --- the GRPO knobs ---
    num_generations=12,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=12,
    max_completion_length=256,
    mask_truncated_completions=False,
    temperature=1.0,
    top_p=0.95,
    top_k=50,
    beta=0.0,
    reward_weights=REWARD_WEIGHTS,
    # --- ordinary training knobs ---
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    num_train_epochs=2,
    bf16=True,
    gradient_checkpointing=True,
    # --- observability ---
    logging_steps=5,
    log_completions=False,
    save_strategy="epoch",
    report_to="none",
)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cuda",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=REWARD_FUNCS,
    args=args,
    train_dataset=train_ds,
    peft_config=peft_config,
)
trainer.train()

### What to watch while it runs

Treat these metrics as diagnostic signals, not definitive diagnoses. Inspect generated SQL and held-out results alongside the training logs.

| Metric | Useful trend | What to check |
| --- | --- | --- |
| `rewards/reward_result_match/mean` | Increasing correctness reward | If flat, inspect generated SQL, component rewards, data, prompt, and optimization settings. |
| `rewards/reward_executes/mean` | Increasing execution success | Low values can indicate syntax errors, unsupported schema references, timeouts, or other execution failures; inspect examples. |
| `frac_reward_zero_std` | Interpret alongside actual rewards | High values indicate many equal-reward groups. Distinguish all-correct from all-wrong groups and check whether rewards discriminate useful differences. |
| `completions/clipped_ratio` | Near zero | Higher values can indicate truncation. Inspect endings, then adjust `max_completion_length` or the prompt. |
| `reward_std` | Enough variation to provide a learning signal | Low variation can reflect consistent success, consistent failure, coarse rewards, or low sampling diversity; inspect completions. |
| `entropy` | Interpret alongside quality and diversity | A sudden drop can indicate narrowing output diversity. Check held-out accuracy and repetitive or degenerate examples before declaring collapse. |

## Did it work?

In [ ]:
import json
import matplotlib.pyplot as plt

# ---- source: live trainer, or a checkpoint's trainer_state.json ----
try:
    LOG = trainer.state.log_history
except NameError:
    ckpt = sorted(
        OUTPUT_DIR.glob("checkpoint-*"),
        key=lambda path: int(path.name.split("-")[-1]),
    )[-1]
    with (ckpt / "trainer_state.json").open(encoding="utf-8") as state_file:
        LOG = json.load(state_file)["log_history"]

def series(key):
    """(steps, values) for one metric, skipping entries that don't carry it."""
    pts = [(e["step"], e[key]) for e in LOG if key in e and isinstance(e[key], (int, float))]
    return [p[0] for p in pts], [p[1] for p in pts]

def ema(v, a=0.15):
    out, s = [], None
    for x in v:
        s = x if s is None else a * x + (1 - a) * s
        out.append(s)
    return out

def draw(ax, key, label, smooth=True, **kw):
    x, y = series(key)
    if not x:
        return False
    ax.plot(x, y, alpha=0.25, linewidth=1, **kw)
    ax.plot(x, ema(y), label=label, linewidth=2, color=ax.lines[-1].get_color())
    return True

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("GRPO run: Taco Alley text-to-SQL", fontsize=13)

# 1. the objective, broken out by reward function
ax = axes[0, 0]
for fn, lbl in [("reward_result_match", "result match"), ("reward_executes", "executes"),
                ("reward_format", "format"), ("reward_row_overlap", "row overlap")]:
    draw(ax, f"rewards/{fn}/mean", lbl)
ax.set_title("Reward components (unweighted)"); ax.set_ylabel("mean reward")
ax.set_ylim(-1.05, 1.05); ax.axhline(0, color="gray", lw=0.5); ax.legend(fontsize=8)

# 2. is there any gradient signal at all?
ax = axes[0, 1]
draw(ax, "reward_std", "reward std")
draw(ax, "frac_reward_zero_std", "frac groups w/ zero std")
ax.set_title("Group disagreement\n(zero-std groups contribute nothing)")
ax.set_ylim(-0.05, 1.35); ax.legend(fontsize=8)

# 3. total reward
ax = axes[0, 2]
draw(ax, "reward", "weighted total")
ax.set_title("Total reward"); ax.legend(fontsize=8)

# 4. length behaviour
ax = axes[1, 0]
draw(ax, "completions/mean_length", "mean length")
ax.set_title("Completion length"); ax.set_ylabel("tokens"); ax.set_xlabel("step")
ax2 = ax.twinx()
x, y = series("completions/clipped_ratio")
if x:
    ax2.plot(x, ema(y), color="tab:red", linewidth=2, label="clipped ratio")
    ax2.set_ylabel("clipped ratio", color="tab:red"); ax2.set_ylim(-0.02, 1.0)
    ax2.legend(fontsize=8, loc="upper right")
ax.legend(fontsize=8, loc="upper left")

# 5. policy health
ax = axes[1, 1]
draw(ax, "entropy", "entropy (nats)")
ax.set_title("Policy entropy\n(a cliff here is collapse)"); ax.set_xlabel("step")
ax2 = ax.twinx()
for k, c in [("clip_ratio/region_mean", "tab:orange"), ("kl", "tab:green")]:
    x, y = series(k)
    if x:
        ax2.plot(x, ema(y), color=c, linewidth=2, label=k)
ax.legend(fontsize=8, loc="upper left")
if ax2.lines:
    ax2.legend(fontsize=8, loc="upper right")

# 6. optimizer view (the GRPO "loss" is a surrogate; grad norm is the useful one)
ax = axes[1, 2]
draw(ax, "grad_norm", "grad norm")
ax.set_title("Optimizer"); ax.set_xlabel("step")
ax2 = ax.twinx()
x, y = series("learning_rate")
if x:
    ax2.plot(x, y, color="tab:gray", linewidth=1.5, linestyle="--", label="lr")
    ax2.set_ylabel("lr", color="tab:gray"); ax2.legend(fontsize=8, loc="upper right")
ax.legend(fontsize=8, loc="upper left")

for a in axes.ravel():
    a.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# quick numeric summary: first vs last decile of steps
keys = ["reward", "rewards/reward_result_match/mean", "rewards/reward_executes/mean",
         "rewards/reward_format/mean", "rewards/reward_row_overlap/mean",
         "`frac_reward_zero_std`", "completions/mean_length",
         "completions/clipped_ratio", "entropy", "reward_std", "grad_norm"]
print(f"{'metric':<40}{'first 10%':>12}{'last 10%':>12}{'delta':>12}")
for k in keys:
    _x, y = series(k)
    if len(y) < 10:
        continue
    n = max(1, len(y) // 10)
    a, b = sum(y[:n]) / n, sum(y[-n:]) / n
    print(f"{k:<40}{a:>12.3f}{b:>12.3f}{b - a:>+12.3f}")

In [ ]:
trained = trainer.model
trained.eval()

after = {"eval_params":    exec_accuracy(trained, tok, eval_params),
         "eval_templates": exec_accuracy(trained, tok, eval_template)}

print(f"{after}")

In [ ]:

print(f"{'split':<18}{'before':>10}{'after':>10}{'delta':>10}")
for k in baseline:
    print(f"{k:<18}{baseline[k]:>9.1%}{after[k]:>10.1%}{after[k]-baseline[k]:>+10.1%}")

In [ ]:
adapter_dir = OUTPUT_DIR / "adapter"
trainer.save_model(str(adapter_dir))
tok.save_pretrained(str(adapter_dir))
print("adapter saved. Merge into the full-precision base model in a fresh process for serving.")